# [Title of Your Project]
**Your name:** [Name]  |  **Starter dataset or your own?** [Starter / Own]

---

## How to use this template
- **Make a copy** (File > Save a copy in Drive) before you start.
- Every section marked **REQUIRED** must be filled in. That's what makes your project complete.
- Sections marked **OPTIONAL** are ways to make your project shine. Skip them if you're short on time.
- Sections 1-3 come straight from your Checkpoint 3 answers. Copy them in and update anything that changed.

### Minimum viable project checklist
A complete project has all of these:
- [ ] One dataset (Section 2)
- [ ] One model compared against a baseline (Sections 5-7)
- [ ] One evaluation metric (Section 7)
- [ ] A fairness check (Section 8)
- [ ] Every REQUIRED section of this template filled in
- [ ] Slides (15 or fewer) and a 5-7 minute video (Section 13)

---
## Agreement for written sections
By filling in the written sections of this notebook (Sections 1, 9, 10, and 11), you confirm that **you did not use AI tools or large language models (LLMs) to write their text**. These sections should be your own reasoning in your own words. AI use anywhere else should be recorded in Section 12.

---
## 1. Research question (REQUIRED)
**Instructions:** In 1-3 sentences, what finance question are you answering, and who would care about the answer?

[Text here]

**What I'm predicting (target variable):** [column name]

**Is this a classification problem (predicting a category) or a regression problem (predicting a number)?** [Classification / Regression]

---
## 2. Dataset (REQUIRED)
**Instructions:** Fill in the table. If you used more than one dataset, add a row for each.

| Dataset name | Link / source | # of rows | Why I chose it |
|---|---|---|---|
| 2025 HMDA Loan/Application Register (LAR), California | https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields | [fill in: final row count printed in Section 3] | Public loan-level mortgage decisions with demographic fields, so outcomes can be compared across groups. |

In [ ]:
# Load your data (HMDA 2025 LAR, California).
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()          # upload your California HMDA CSV
CSV_PATH = "state_CA.csv"          # <- must match the uploaded file name

KEEP = [
    # columns used only to filter rows
    "activity_year", "state_code", "action_taken", "loan_purpose", "occupancy_type",
    "business_or_commercial_purpose", "lien_status", "construction_method",
    "total_units", "reverse_mortgage", "open-end_line_of_credit",
    # model features
    "income", "debt_to_income_ratio", "loan_amount", "loan_to_value_ratio",
    "property_value", "loan_type",
    # fairness analysis only (never model inputs) + optional lender id
    "derived_race", "derived_ethnicity", "derived_sex", "applicant_age", "lei",
]
rename = {"combined_loan_to_value_ratio": "loan_to_value_ratio"}   # official name -> project name
header = pd.read_csv(CSV_PATH, nrows=0).columns
usecols = [c for c in header if rename.get(c, c) in KEEP]
raw = pd.read_csv(CSV_PATH, dtype=str, low_memory=False, usecols=usecols).rename(columns=rename)
df = raw

print("Dataset shape:", raw.shape)
print("Missing expected columns:", [c for c in KEEP if c not in raw.columns])
df.head()

---
## 3. Data cleaning (REQUIRED)
**Instructions:** Clean your data in the code cell below (for example: fill or drop missing values, drop columns you don't need, convert text categories to numbers). Then list what you changed and why.

1. Kept only California 2025 home-purchase applications for a principal residence, not for business use, first-lien, site-built, 1-4 units, not reverse mortgages or lines of credit, with an outcome of originated, approved-not-accepted, or denied (`action_taken` 1, 2, 3). This makes every row a comparable owner-occupant purchase decision.
2. Created the target `approved`: 1 = originated or approved-not-accepted, 0 = denied.
3. Converted income, loan amount, loan-to-value ratio and property value to numbers and dropped rows where they were missing or not positive.
4. Converted debt-to-income ranges to numeric midpoints. `NA`/`Exempt` rows are kept with a `dti_missing` flag and filled with the training-set median after the split.
5. Left out columns that only exist because of the decision (denial reasons, pricing and fee fields, purchaser type) to avoid data leakage.
6. Kept race, ethnicity and sex out of the model inputs, and kept applicant age out too because it is also a protected characteristic. These columns are used only in the fairness check.

In [ ]:
# 3a. Row filters (printed as a funnel -> use these numbers in Section 2/3)
filters = [
    ("year 2025",                        lambda d: d.activity_year == "2025"),
    ("California",                       lambda d: d.state_code == "CA"),
    ("action_taken in 1,2,3",            lambda d: d.action_taken.isin(["1", "2", "3"])),
    ("home purchase",                    lambda d: d.loan_purpose == "1"),
    ("principal residence",              lambda d: d.occupancy_type == "1"),
    ("not business/commercial",          lambda d: d.business_or_commercial_purpose == "2"),
    ("first lien",                       lambda d: d.lien_status == "1"),
    ("site-built",                       lambda d: d.construction_method == "1"),
    ("1-4 units",                        lambda d: d.total_units.isin(["1", "2", "3", "4"])),
    ("not reverse mortgage",             lambda d: d.reverse_mortgage == "2"),
    ("not open-end line of credit",      lambda d: d["open-end_line_of_credit"] == "2"),
]
df = raw
print(f"{'start':35s}{len(df):>12,}")
for name, f in filters:
    df = df[f(df)]
    print(f"{name:35s}{len(df):>12,}")
df = df.copy()

# 3b. Target: 1 = approved (actions 1,2), 0 = denied (action 3)
df["approved"] = df.action_taken.isin(["1", "2"]).astype(int)
print("Denial rate after filters: %.2f%%" % (100 * (1 - df.approved.mean())))

# 3c. Numeric columns
num_cols = ["income", "loan_amount", "loan_to_value_ratio", "property_value"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 3d. Debt-to-income: bins -> numeric midpoints, NA/Exempt -> missing flag
def dti_to_num(v):
    if pd.isna(v):
        return np.nan
    v = str(v).strip().replace("%", "")
    bins = {"<20": 15, "20-<30": 25, "30-<36": 33, "50-60": 55, ">60": 65}
    if v in bins:
        return bins[v]
    try:
        return float(v)
    except ValueError:        # "NA", "Exempt", ...
        return np.nan

df["dti_num"] = df.debt_to_income_ratio.map(dti_to_num)
df["dti_missing"] = df.dti_num.isna().astype(int)

# 3e. LEAKAGE CHECK: is missingness related to the outcome? (share missing, by outcome)
check = df[num_cols + ["dti_num"]].isna().groupby(df.approved).mean().round(3)
check.index = check.index.map({0: "denied", 1: "approved"})
print(check)

# 3f. Drop rows missing core numbers or with non-positive values (DTI missing is KEPT, flagged)
n0 = len(df)
df = df.dropna(subset=num_cols)
df = df[(df.income > 0) & (df.loan_amount > 0) & (df.property_value > 0) & (df.loan_to_value_ratio > 0)]
print(f"Dropped {n0 - len(df):,} rows; remaining {len(df):,}")
print("Final denial rate: %.2f%%" % (100 * (1 - df.approved.mean())))

df.info()

---
## 4. Exploring the data (REQUIRED)
**Instructions:** Include at least one chart from your EDA (you can reuse your Checkpoint 3 work). Under it, write 1-2 sentences about what it shows.

[What the chart shows]

In [ ]:
t = (df.assign(denied=1 - df.approved)
       .groupby("derived_race").agg(n=("denied", "size"), denied=("denied", "sum"), rate=("denied", "mean"))
       .sort_values("rate"))
print(t)
t = t[t.denied >= 30]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([f"{i} (n={n:,})" for i, n in zip(t.index, t.n)], t.rate * 100)
ax.axvline(100 * (1 - df.approved.mean()), color="k", ls="--", label="overall")
ax.set_xlabel("Unadjusted denial rate (%)")
ax.set_title("Denial rate by race group (groups with >= 30 denials)")
ax.legend(); plt.tight_layout(); plt.show()

---
## 5. Train/test split and baseline (REQUIRED)
A **baseline** is the simplest possible guess. If your model can't beat it, the model isn't adding much. Run the cell below as-is after filling in your features and target.
- For **classification**, the baseline always predicts the most common category.
- For **regression**, the baseline always predicts the average.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

features_num = ["income", "dti_num", "dti_missing", "loan_amount", "loan_to_value_ratio", "property_value"]
X = pd.concat([df[features_num], pd.get_dummies(df.loan_type, prefix="loan_type", dtype=int)], axis=1)
y = df.approved
meta = df[["derived_race", "derived_ethnicity", "derived_sex", "applicant_age", "lei"]]   # NOT in X

# stratify on outcome x race (rare cells fall back to outcome only)
strat = y.astype(str) + "_" + df.derived_race.fillna("NA")
cnt = strat.map(strat.value_counts())
strat = strat.where(cnt >= 10, y.astype(str))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=strat)

# fill DTI gaps with the TRAIN median (the dti_missing flag keeps the information)
dti_med = X_train["dti_num"].median()
X_train = X_train.fillna({"dti_num": dti_med})
X_test = X_test.fillna({"dti_num": dti_med})
meta_test = meta.loc[X_test.index]

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape, "| test denial rate: %.2f%%" % (100 * (1 - y_test.mean())))
print("Features:", list(X.columns))

---
## 6. Model (REQUIRED: at least one)
**Instructions:** Train at least one model you learned in the challenge (linear regression, logistic regression, decision tree, or random forest). Then fill in the summary.

#### Model 1
* **Model type:** Random Forest classifier (300 trees, `min_samples_leaf=20`, `class_weight="balanced_subsample"`), with the denial-probability threshold chosen on a validation split.
* **Features used:** income, debt-to-income (numeric + missing flag), loan amount, loan-to-value ratio, property value, loan type. Race, ethnicity and sex are NOT inputs.
* **Why I chose this model:** it handles non-linear effects and interactions between underwriting factors without scaling, and it works with SHAP for explanations. Class weights address that only about 1 in 9 applications is denied.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# validation split (inside train) used ONLY to choose the decision threshold
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

def fit_with_threshold(model, Xtr, ytr, Xval, yval, Xte):
    """Fit, pick the denial-probability threshold that maximizes denial-F1 on validation, apply to test."""
    model.fit(Xtr, ytr)
    p_val = model.predict_proba(Xval)[:, 0]          # column 0 = P(denied)
    ths = np.arange(0.05, 0.96, 0.01)
    f1s = [f1_score((yval == 0).astype(int), (p_val >= t).astype(int), zero_division=0) for t in ths]
    t_best = float(ths[int(np.argmax(f1s))])
    p_test = model.predict_proba(Xte)[:, 0]
    return model, t_best, p_test, np.where(p_test >= t_best, 0, 1)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced_subsample",
                            n_jobs=-1, random_state=42)
rf, t_rf, p_rf, y_pred = fit_with_threshold(rf, X_tr, y_tr, X_val, y_val, X_test)

lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
lr, t_lr, p_lr, y_pred_lr = fit_with_threshold(lr, X_tr, y_tr, X_val, y_val, X_test)
print(f"RF threshold on P(denied): {t_rf:.2f} | LR threshold: {t_lr:.2f}  (fixed for ALL groups)")

#### Model 2 (OPTIONAL)
* **Model type:** Logistic regression (standardized inputs, `class_weight="balanced"`), trained in the same cell as Model 1 with the same threshold procedure.
* **Features used:** same as Model 1.
* **Why I chose this model:** a simple interpretable benchmark. If the random forest does not clearly beat it, the extra complexity is not adding much.

---
## 7. Evaluation (REQUIRED)
**Instructions:** Pick one metric that fits your question and compare your model to the baseline.
- Classification: accuracy, precision, recall, or F1. If one category is rare (like fraud or default), accuracy can be misleading, so also look at recall or F1.
- Regression: RMSE or R-squared.

| Model | Metric: denial recall / F1 (denied = positive class) |
|---|---|
| Baseline | [value] |
| Model 1 | [value] |

**Does your model beat the baseline? By how much, and does that difference matter for your question?**

[Text here]

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

def eval_row(name, y_true, y_hat, p_deny=None):
    d_true = (np.asarray(y_true) == 0).astype(int)
    d_hat = (np.asarray(y_hat) == 0).astype(int)
    row = {"Model": name,
           "Accuracy": accuracy_score(y_true, y_hat),
           "Denial precision": precision_score(d_true, d_hat, zero_division=0),
           "Denial recall": recall_score(d_true, d_hat, zero_division=0),
           "Denial F1": f1_score(d_true, d_hat, zero_division=0)}
    if p_deny is not None:
        row["ROC-AUC"] = roc_auc_score(d_true, p_deny)
        row["PR-AUC"] = average_precision_score(d_true, p_deny)
    return row

results = pd.DataFrame([
    eval_row("Baseline (always approve)", y_test, baseline_pred),
    eval_row("Model 1: Random Forest", y_test, y_pred, p_rf),
    eval_row("Model 2: Logistic Regression", y_test, y_pred_lr, p_lr),
]).set_index("Model").round(3)
print(results)
print("\nPR-AUC no-skill reference (= test denial rate): %.3f" % (1 - y_test.mean()))

---
## 8. Fairness check (REQUIRED)
**Instructions:** Choose one column that describes a group of people (for example: sex, age group, race, marital status). Check whether your model performs equally well for each group. You don't need to include that column as a feature to check it.

**Group column I checked:** `derived_race` (also `derived_ethnicity` and `derived_sex`)

**What I found (2-3 sentences):** Is the model noticeably better or worse for any group? Why might that be, and who could be harmed?

[Text here]

In [ ]:
def group_table(groups, y_true, y_hat, min_denials=30, n_boot=200, seed=0):
    """Per-group denial metrics with bootstrap 95% CIs. Positive class = denial."""
    rng = np.random.default_rng(seed)
    g_arr, yt, yh = np.asarray(groups), np.asarray(y_true), np.asarray(y_hat)
    rows = []
    for g in pd.unique(g_arr[~pd.isna(g_arr)]):
        m = g_arr == g
        d_true, d_hat = (yt[m] == 0), (yh[m] == 0)
        if d_true.sum() < min_denials:
            continue
        def stats(idx):
            dt, dh = d_true[idx], d_hat[idx]
            return (dt.mean(), dh.mean(),
                    (dh & dt).sum() / max(dt.sum(), 1),        # denial recall
                    (dh & ~dt).sum() / max((~dt).sum(), 1))    # approved applicants flagged as denial
        n = m.sum()
        point = stats(np.arange(n))
        boots = np.array([stats(rng.integers(0, n, n)) for _ in range(n_boot)])
        lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
        rows.append({"group": g, "n": n, "actual denials": int(d_true.sum()),
                     "actual denial rate": point[0], "predicted denial rate": point[1],
                     "denial recall": f"{point[2]:.2f} [{lo[2]:.2f}-{hi[2]:.2f}]",
                     "approved wrongly flagged": f"{point[3]:.2f} [{lo[3]:.2f}-{hi[3]:.2f}]"})
    return pd.DataFrame(rows).set_index("group").sort_values("n", ascending=False).round(3)

for col in ["derived_race", "derived_ethnicity", "derived_sex"]:
    print(f"\n=== Random Forest (race-blind) by {col} ===")
    print(group_table(meta_test[col], y_test, y_pred))

---
## 9. Limitations (REQUIRED)
**Instructions:** In 3-5 sentences: What can't your data or model tell you? What's missing from the data, and what would you do with more time?

[Text here]

---
## 10. Results (REQUIRED)
**Instructions:** In 5-10 sentences, explain what you found. Refer to your metric, the baseline, and your fairness check.

[Text here]

---
## 11. Conclusion and recommendations (REQUIRED)
**Instructions:** In 5-8 sentences: What should a bank, lender, regulator, or customer do with your findings?

[Text here]

---
## 12. AI-use log (REQUIRED)
**Instructions:** List each way you used AI tools (chatbots, code assistants, etc.). If you didn't use any, write "None."

| Tool | What I used it for | How I checked the output |
|---|---|---|
| Claude (Anthropic) | Reviewed my research design: dataset filters, target definition, feature selection, and which HMDA columns could leak the outcome. | Compared the column names and codes against the official FFIEC HMDA LAR data fields documentation; printed the filter funnel and the missingness-by-outcome table in this notebook. |
| Claude (Anthropic) | Suggested fairness metrics and an evaluation approach for an imbalanced target (recall/F1 on denials, per-group error rates, bootstrap intervals). | Read what each metric measures and checked the group counts and denial counts printed in Section 8. |
| Claude (Anthropic) | Wrote the starter code for Sections 2-8 and the optional cells (data cleaning, split, models, evaluation, fairness table, race-aware comparison, adjusted-gap regression, SHAP). | Ran every cell myself in Colab, read the outputs, and fixed errors. Sections 1, 9, 10 and 11 were written by me without AI. |

---
## 13. Slides, video, and references (REQUIRED)
- **Slides (15 or fewer):** [link]
- **Video (5-7 minutes):** [link]

**References:** Links to any articles or resources you used to interpret results or make recommendations.
1. [Reference 1 with link]
2. [...]

---
## Make it shine (OPTIONAL)
Ideas if you have extra time: compare 2-3 models, tune a model (for example, `max_depth`), show feature importance, combine two datasets, or add a confusion matrix. See "Tips to Make Your Project Shine" on the challenge homepage.

### Optional A: race-aware vs race-blind model
Does adding race to the inputs change predictions or performance? If not, the other features already carry the same information.

In [ ]:
race_d = pd.get_dummies(df.derived_race, prefix="race", dtype=int)
Xa_tr = pd.concat([X_tr, race_d.loc[X_tr.index]], axis=1)
Xa_val = pd.concat([X_val, race_d.loc[X_val.index]], axis=1)
Xa_te = pd.concat([X_test, race_d.loc[X_test.index]], axis=1)

rf_aware = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced_subsample",
                                  n_jobs=-1, random_state=42)
rf_aware, t_a, p_a, y_pred_aware = fit_with_threshold(rf_aware, Xa_tr, y_tr, Xa_val, y_val, Xa_te)

print(pd.DataFrame([eval_row("RF race-blind", y_test, y_pred, p_rf),
                    eval_row("RF race-aware", y_test, y_pred_aware, p_a)]).set_index("Model").round(3))

cmp_ = pd.concat({"blind": group_table(meta_test.derived_race, y_test, y_pred, n_boot=1)["predicted denial rate"],
                  "aware": group_table(meta_test.derived_race, y_test, y_pred_aware, n_boot=1)["predicted denial rate"],
                  "actual": group_table(meta_test.derived_race, y_test, y_pred, n_boot=1)["actual denial rate"]}, axis=1)
print("\nPredicted denial rate by race: blind vs aware vs actual")
print(cmp_)

### Optional B: adjusted gap (logistic regression)
Odds ratios for denial by race, holding the listed underwriting factors constant. Credit score, assets and lender are not observed, so this is not proof of discrimination.

In [ ]:
import statsmodels.formula.api as smf
adj = df.copy()
adj["denied"] = 1 - adj.approved
adj["dti_f"] = adj.dti_num.fillna(adj.dti_num.median())
adj["log_income"] = np.log(adj.income)
adj["log_loan"] = np.log(adj.loan_amount)
adj["log_value"] = np.log(adj.property_value)
major = adj.derived_race.value_counts()
adj = adj[adj.derived_race.isin(major[major >= 200].index) & (adj.derived_race != "Race Not Available")]
fit = smf.logit("denied ~ C(derived_race, Treatment(reference='White')) + log_income + dti_f + dti_missing"
                " + log_loan + loan_to_value_ratio + log_value + C(loan_type)", data=adj).fit(disp=0)
orr = np.exp(fit.params).rename("odds ratio").to_frame().join(np.exp(fit.conf_int()).rename(columns={0: "lo", 1: "hi"}))
print(orr[orr.index.str.contains("derived_race")].round(2))
print("\nOdds ratio > 1 = higher odds of denial than White applicants with the same listed characteristics.")
print("Credit score, assets, lender are NOT observed -> this is NOT proof of discrimination.")

### Optional C: SHAP
Which features push the race-blind random forest toward approval or denial, overall and by group.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=False)
import shap

Xs = X_test.sample(min(3000, len(X_test)), random_state=42)
sv = shap.TreeExplainer(rf).shap_values(Xs)
sv_a = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if np.ndim(sv) == 3 else sv)   # class 1 = approved

shap.summary_plot(sv_a, Xs, show=False); plt.title("SHAP: push toward APPROVAL (+) or DENIAL (-)"); plt.show()
shap.summary_plot(sv_a, Xs, plot_type="bar", show=False); plt.show()
shap.dependence_plot("dti_num", sv_a, Xs, show=False); plt.show()

# mean SHAP by race group: which features pull each group's predicted approval up/down
sv_df = pd.DataFrame(sv_a, columns=Xs.columns, index=Xs.index)
grp = meta.loc[Xs.index, "derived_race"]
ok = grp.map(grp.value_counts()) >= 50
print(sv_df[ok].groupby(grp[ok]).mean().round(4))
print("\nSHAP explains the MODEL, not causes. Groups differ because their inputs differ.")